# Module A — HCC / RADV Risk Flag Analysis

**Track:** RADV / Risk Adjustment  
**Path:** `src/modules/hcc_radv_risk_flags/`

This notebook applies RADV audit targeting logic to the MSSP PUF. The PUF risk score columns (by enrollment type and data cut) are the analytical input. The module reverse-engineers which HCC groupings drive risk score variance, flags counties with patterns consistent with RADV audit risk indicators, and produces a ranked risk exposure table.

**Key finding target:** Identify top counties where risk score growth YoY exceeded 2 SD and expenditure growth exceeded the national trend factor. Cross-reference with HCC concentration — DM+CHF, CHF+CKD combinations are RADV audit priority codes.

---
**Benchmarks used:**
- CMS national trend factor: 0–3% annual risk score growth in stable populations  
- RADV flag threshold: counties exceeding mean + 2 SD YoY delta  
- HCC concentration flag: top-2 HCC share > 60%  
- RADV exposure composite: top 10% by weighted score

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from hcc_mapper import estimate_hcc_concentration_proxy, HCC_TO_CATEGORY
from risk_score_variance import compute_risk_score_yoy_delta
from radv_proxy_flags import compute_radv_exposure_score
from hcc_risk_report import build_hcc_risk_flag_summary

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Synthetic data

Replace `df` with your MSSP PUF dataframe loaded via `DataIngestionPipeline.run()`. The synthetic data below mirrors the PUF schema and V24/V28 transition pattern: a subset of counties shows elevated risk score growth in 2024 reflecting the HCC upweighting pattern.

In [ ]:
rng = np.random.default_rng(42)
n_counties = 60
states = [f"{i:02d}" for i in range(1, 13)]
counties = [f"{i:03d}" for i in range(1, 6)]
enrollment_types = ["Aged Non-Dual", "Aged Dual", "Disabled", "ESRD"]
years = ["2021", "2022", "2023", "2024"]

rows = []
for state in states:
    for county in counties:
        base_risk = rng.uniform(0.85, 1.40)
        base_exp  = rng.uniform(9_000, 16_000)
        py        = rng.integers(200, 8_000)
        for et in ["Aged Non-Dual", "Aged Dual"]:
            for i, yr in enumerate(years):
                # Simulate V28 transition uplift in 2024 for a random 30% of counties
                v28_bump = 1.08 if (rng.random() < 0.30 and yr == "2024") else 1.0
                rows.append({
                    "year":             yr,
                    "state_id":         state,
                    "county_id":        county,
                    "state_name":       f"State {state}",
                    "county_name":      f"County {county}",
                    "enrollment_type":  et,
                    "avg_risk_score":   round(base_risk * (1.01 ** i) * v28_bump + rng.normal(0, 0.03), 4),
                    "per_capita_exp":   round(base_exp  * (1.02 ** i) + rng.normal(0, 300), 2),
                    "person_years":     int(py * (1.005 ** i)),
                    "data_cut":         "OC3",
                    "dataset_id":       "7c34-eaqd",
                })

df = pd.DataFrame(rows)
df["expenditure_efficiency_ratio"] = df["per_capita_exp"] / df["avg_risk_score"]
print(f"Synthetic dataset: {len(df):,} rows  |  {df['state_id'].nunique()} states  |  {df['year'].unique().tolist()} years")
df.head()

## 2. Year-over-year risk score delta

In [ ]:
df = compute_risk_score_yoy_delta(df)

# Outlier detection: counties exceeding mean + 2 SD
yoy_2024 = df.loc[df["year"] == "2024", "risk_score_yoy_delta"].dropna()
mean_delta = yoy_2024.mean()
std_delta  = yoy_2024.std()
threshold_2sd = mean_delta + 2 * std_delta

print(f"2024 YoY risk score delta — Mean: {mean_delta:.4f}  |  SD: {std_delta:.4f}  |  +2 SD threshold: {threshold_2sd:.4f}")
print(f"Counties flagged (> mean + 2 SD): {(yoy_2024 > threshold_2sd).sum()}")

## 3. Distribution of YoY risk score delta — 2024

In [ ]:
plot_df = df.loc[df["year"] == "2024"].dropna(subset=["risk_score_yoy_delta"])

fig = px.histogram(
    plot_df,
    x="risk_score_yoy_delta",
    nbins=40,
    color="enrollment_type",
    barmode="overlay",
    opacity=0.7,
    title="Distribution of YoY Risk Score Delta — 2024 (OC3)",
    labels={"risk_score_yoy_delta": "YoY Risk Score Delta", "enrollment_type": "Enrollment Type"},
)
fig.add_vline(x=threshold_2sd, line_dash="dash", line_color="red",
              annotation_text=f"+2 SD ({threshold_2sd:.3f})", annotation_position="top right")
fig.add_vline(x=mean_delta, line_dash="dot", line_color="gray",
              annotation_text="Mean", annotation_position="bottom right")
fig.update_layout(legend_title_text="Enrollment Type")
fig.show()

## 4. Build full RADV exposure flag table

In [ ]:
summary = build_hcc_risk_flag_summary(df, top_n=30)
print(f"Top {len(summary)} RADV-flagged county-enrollment records")
summary[["year", "state_id", "county_id", "enrollment_type",
         "avg_risk_score", "risk_score_yoy_delta",
         "expenditure_efficiency_ratio", "hcc_concentration_index",
         "radv_exposure_score", "radv_exposure_flag"]].style.background_gradient(
    subset=["radv_exposure_score"], cmap="YlOrRd"
).format({
    "avg_risk_score":              "{:.4f}",
    "risk_score_yoy_delta":        "{:.3%}",
    "expenditure_efficiency_ratio":"{:.0f}",
    "hcc_concentration_index":     "{:.3f}",
    "radv_exposure_score":         "{:.4f}",
})

## 5. RADV exposure scatter — YoY delta vs. expenditure efficiency

In [ ]:
scatter_df = df.loc[
    df["year"] == "2024"
].dropna(subset=["risk_score_yoy_delta", "expenditure_efficiency_ratio"])

# Compute exposure score for the scatter subset
scatter_df = compute_radv_exposure_score(scatter_df)

fig = px.scatter(
    scatter_df,
    x="risk_score_yoy_delta",
    y="expenditure_efficiency_ratio",
    color="radv_exposure_flag",
    color_discrete_map={True: "#d62728", False: "#1f77b4"},
    size="hcc_concentration_index",
    size_max=14,
    hover_data=["state_id", "county_id", "enrollment_type", "radv_exposure_score"],
    title="RADV Exposure Proxy: YoY Risk Score Delta vs. Expenditure Efficiency (2024)",
    labels={
        "risk_score_yoy_delta":         "YoY Risk Score Delta",
        "expenditure_efficiency_ratio":  "Expenditure Efficiency Ratio ($/RAF)",
        "radv_exposure_flag":            "High RADV Exposure",
    },
)
fig.add_vline(x=threshold_2sd, line_dash="dash", line_color="red", opacity=0.5)
fig.update_layout(legend_title_text="High RADV Exposure")
fig.show()

## 6. Key finding

> Counties with the highest RADV exposure scores combine above-average risk score increases, concentrated risk patterns, and above-average expenditure efficiency. The 2024 data shows a subset of counties with YoY risk score growth exceeding +2 SD of the national distribution — a pattern consistent with V28 HCC upweighting rather than genuine population health deterioration. These records should be prioritised for review in MSSP payment integrity or MA audit modelling workflows, particularly where DM+CHF or CHF+CKD combinations are the primary drivers (RADV audit priority code pairs per CMS 2019 RADV methods documentation).